In [1]:
%load_ext autoreload
import sys
from datetime import datetime
from ptflops import get_model_complexity_info
from torch.optim import Adam
import torch
from torch import nn

sys.path.append("..")
%autoreload 2
from src import *

sns.set_theme("paper", font_scale=2.5)

In [2]:
df_x, df_y = read_dataset(stage="original")
# df_x['np/ng'] = df_x['np'] / df_x['ng']
df_x.drop(inplace=True, columns=["ias", "oat"])
df_x["trq_margin"] = df_y["trq_margin"]
df_x["task"] = (df_x["np"] / df_x["ng"] < 1).astype(int) + 1

In [3]:
def get_title(model, epochs, times, lr, is_file=False):
    additional = f"grid={model.grid_size}" if type(model) in [PyKAN, EfficientKAN] else ""
    return (
        f'{model.__class__.__name__}_layers=[{"-".join(map(str, model.layers))}]_epochs={epochs}_times={times}_lr={lr}_{additional}_{datetime.now().ctime().replace(":", "-")}'
        if is_file
        else f"{model.__class__.__name__} Epochs={epochs} Times={times} Lr={lr}"
    )

In [4]:
def get_criterion(task: int):
    def criterion(x: torch.Tensor, y: torch.Tensor):
        return nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2]).to(x.device))(x[:, task - 1], y.squeeze())

    return criterion


def itl_train(
    model: PHMNetwork,
    tasks: list[int],
    epochs: int,
    times: int,
    lr=1e-3,
    device="cpu",
    silent=True,
) -> tuple[list, list, list]:
    model.reset()
    train_losses, valid_losses, test_metrics = [], [], []
    optimizer = Adam(model.parameters(), lr=lr)
    trainset, validset, testset, _ = split_dataset(
        df_x,
        df_y["faulty"],
        0.5,
        standardize_y=False,
        group_by="task",
        group_testset=True,
        device=device,
        seed=0,
    )
    for i, task in enumerate(tqdm(tasks * times)):
        min_length = min(trainset[1][0].shape[0], trainset[2][0].shape[0])
        for t in tasks:
            trainset[t] = (trainset[t][0][:min_length], trainset[t][1][:min_length])

        def test():
            metrics = {}
            for t in tasks:
                metrics[t] = model.test(
                    testset[t],
                    batch_size=2048,
                    silent=silent,
                    task=t,
                )
                metrics[t]["samples"]=len(testset[t])
                
            test_metrics[i].append(metrics)

        test_metrics.append([])
        train_loss, valid_loss = model.fit(
            trainset[task],
            validset[task],
            optimizer,
            epochs=epochs,
            batch_size=2048,
            callback=test,
            silent=silent,
            criterion=get_criterion(task=task),
        )
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        # model.save(f'IDL_{model.__class__.__name__}_{domain}')

    # Save to file
    with open(f"results/itl/{get_title(model, epochs, times, lr, is_file=True)}", "wb") as f:
        pickle.dump((train_losses, valid_losses, test_metrics), f)
    return train_losses, valid_losses, test_metrics

In [6]:
TASKS = [1, 2]
EPOCHS = 5
TIMES = 1
LR = 5e-2

## Finding a matching shape for KAN and MLP with `ptflops`

In [43]:
kan_shape = (len(df_x.columns) - 1, 51, 51, 1)
mlp_shape = (len(df_x.columns) - 1, 256, 256, 1)

effKAN = EfficientKAN([*kan_shape], "classification", grid_size=20, continual_learning=True)
mlp = MLP([*mlp_shape], "classification")

for net in [effKAN, mlp]:
    flops, params = get_model_complexity_info(
        net, (len(df_x.columns) - 1,), as_strings=True, print_per_layer_stat=False
    )
    print(f"[{net.__class__.__name__}] FLOPs: {flops} | Params: {params}")

[EfficientKAN] FLOPs: 108 Mac | Params: 68.03 k
[MLP] FLOPs: 67.84 KMac | Params: 67.84 k


# KAN

In [38]:
effKAN = EfficientKAN(
    kan_shape,
    "classification",
    grid_size=20,
    continual_learning=True,
    device="cuda",
)
itl_train(effKAN, TASKS, epochs=EPOCHS, times=TIMES, lr=LR, device="cuda")
pass

  0%|          | 0/2 [00:00<?, ?it/s]/home/vmorelli-iit.local/Projects/PHM_North_America_2024_Challenge/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vmorelli-iit.local/Projects/PHM_North_America_2024_Challenge/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
100%|██████████| 2/2 [00:46<00:00, 23.21s/it]


# MLP

In [39]:
mlp = MLP(mlp_shape, "classification", device="cuda")
itl_train(mlp, TASKS, epochs=EPOCHS, times=TIMES, lr=LR, device="cuda")
pass

100%|██████████| 2/2 [00:03<00:00,  1.55s/it]


# Visualize the results with the visualizer GUI

In [9]:
TITLES = [
    "MLP vs KAN without Replay (TIL)",
    "MLP vs KAN with Replay (TIL)",
]
continual_learning_gui(
    results_dir=Path("results/itl/").absolute(),
    metric="avg_test_score",
    ylim=[0.25, 1],
    title=TITLES[0],
    tasks=TASKS,
    loc="lower right"
)

In [53]:
exp="MLP_layers=[6-256-256-2]_epochs=5_times=8_lr=0.02__Fri Jan  9 14-42-53 2026"



f=Path(f"results/itl/{exp}")
metrics=pickle.load(f.open("rb"), encoding="utf-8")[2]
loss=[sum(x[t]["avg_test_score"] * x[t]["samples"] for t in [1,2]) / sum(x[t]["samples"] for t in [1,2]) for xs in metrics for x in xs ]
max(loss), loss.index(max(loss)), loss

(0.9556035249546369,
 74,
 [0.3281551300556522,
  0.3675066646669267,
  0.37477718112478436,
  0.3809936567522898,
  0.39476384655252367,
  0.7341555455253511,
  0.7325728056709917,
  0.731673766781786,
  0.731200139128854,
  0.7309895114258719,
  0.8015599639418343,
  0.8194088856946486,
  0.8381973613182312,
  0.848233794189704,
  0.856792754421507,
  0.8521633169564489,
  0.8525311116353309,
  0.8557370074705934,
  0.8587396105208379,
  0.8628587524886644,
  0.9131692469338877,
  0.9191043652977747,
  0.920110676451543,
  0.9204491857158459,
  0.9203535447984539,
  0.9031900281246221,
  0.9036159242660426,
  0.9049876645411399,
  0.9059056523463659,
  0.9066156778626246,
  0.9361061986222794,
  0.937414440858657,
  0.9386747909219015,
  0.9398293690559143,
  0.9408931509239977,
  0.9212492923649466,
  0.9188760504244233,
  0.9167823655021544,
  0.9153012355611362,
  0.9151065392826516,
  0.9420396592158713,
  0.943169838187474,
  0.9434394022421233,
  0.9437389554390693,
  0.9445009